In [5]:
import cv2
import matplotlib
import numpy as np
from shapely import Polygon

SPEED_THRESHOLD = 3
AREA_THRESHOLD = 2000
MARGIN = 5
DELTA_T = 5
N_COLORS = 9

# Read in a video file
vidReader = cv2.VideoCapture("visiontraffic.avi")

# Skip first still frames
for i in range(90):
    ret, frame = vidReader.read()

# Initialize optical flow
prevGray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

# Get the frame width and height
frame_width = int(vidReader.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(vidReader.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Define the codec and create a VideoWriter object
out = cv2.VideoWriter(
    "output.avi", cv2.VideoWriter_fourcc(*"XVID"), 20.0, (frame_width, frame_height)
)

frame_i = 0

detections = []
existing_paths = []
current_i = 0
colors = matplotlib.cm.Set1(range(N_COLORS))


def get_detection(contour: np.ndarray) -> dict:
    try:
        poly = Polygon(contour)
    except:
        return None
    for existing in detections:
        if frame_i - existing["last_seen"] > DELTA_T:
            continue
        existing_poly = Polygon(existing["contour"])
        intersection = poly.intersection(existing_poly)
        union = poly.union(existing_poly)
        if intersection.area / union.area > 0.1:
            return existing
    return None


while vidReader.isOpened():
    ret, frameRGB = vidReader.read()
    if not ret:
        break
    frameGray = cv2.cvtColor(frameRGB, cv2.COLOR_BGR2GRAY)

    # Estimate optical flow using Farneback method
    flow = cv2.calcOpticalFlowFarneback(
        prevGray, frameGray, None, 0.5, 3, 15, 3, 5, 1.2, 0
    )

    # Calculate speed and direction from Vx and Vy
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])

    mask = mag > SPEED_THRESHOLD
    bin = mask.astype(np.uint8)
    bin = cv2.morphologyEx(bin, cv2.MORPH_DILATE, (10, 10))
    mask = bin > 0

    # Remove all measurements for speeds lower than SPEED_THRESHOLD
    filt_dir = np.zeros_like(ang)
    filt_dir[mask] = ang[mask]

    # Calculate region statistics for thresholded image
    contours, _ = cv2.findContours(bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Analyze found regions
    annotations = []

    for contour in contours:
        contour_2d = contour.reshape(-1, 2)
        area = cv2.contourArea(contour)
        if area > AREA_THRESHOLD:
            bb = cv2.boundingRect(contour)
            x, y, w, h = bb

            dirbb = filt_dir[y : y + h, x : x + w]
            spdbb = mag[y : y + h, x : x + w]
            det_dir = np.mean(dirbb[dirbb != 0])
            det_spd = np.mean(spdbb[spdbb > 2])
            cx, cy = np.mean(contour, axis=0)[0]

            # if cx < MARGIN or frame_width - cx < MARGIN or cy < MARGIN or frame_height - cy < MARGIN:
            #     continue

            detection = {
                "ok": True,
                "bb": bb,
                "dir": det_dir,
                "spd": det_spd,
                "contour": contour_2d,
                "cc": (cx, cy),
                "lbl": area,
                "last_seen": frame_i,
            }

            existing = get_detection(contour_2d)
            if existing is None:
                detection["i"] = current_i
                current_i += 1
                detections.append(detection)
                result = detection
            else:
                i = existing["i"]
                detections[i] |= detection
                result = detections[i]

            annotations.append(result)

    # Draw annotations (bounding boxes with the segment area)
    ann = frameRGB.copy()
    for det in annotations:
        x, y, w, h = det["bb"]
        cv2.rectangle(ann, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(
            ann,
            str(int(det["i"])),
            (x + w, y + 15),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0, 255, 0),
            2,
        )

    # Draw the average direction vector
    for det in annotations:
        x1, y1 = det["cc"]
        x2 = int(x1 + 5 * det["spd"] * np.cos(det["dir"]))
        y2 = int(y1 + 5 * det["spd"] * np.sin(det["dir"]))
        cv2.arrowedLine(ann, (int(x1), int(y1)), (x2, y2), (0, 0, 255), 3)

    # Add paths
    for det in annotations:
        x1, y1 = det["cc"]
        i = det["i"]
        color = colors[i % N_COLORS]
        color = tuple(map(int, color[2::-1] * 255))
        current_path = (color, (int(x1), int(y1)))
        existing_paths.append(current_path)

    # Draw paths
    for color, path in existing_paths:
        cv2.circle(ann, path, 4, color, -1)

    # Write the annotated frame to the output video
    out.write(ann)

    # Update previous frame
    prevGray = frameGray

    frame_i += 1

vidReader.release()
out.release()

In [6]:
!ffmpeg -y -i output.avi output.mp4 -hide_banner -loglevel error

In [7]:
from IPython.display import Video

Video("output.mp4", embed=True)